# V2 — 2D U-Net Training

This notebook trains the V2 baseline for multimodal 3D brain tumor MRI segmentation.

### Experiment
- Input: 4 MRI modalities — T1n, T1c, T2w, T2f
- Resolution: `128×128`
- Task: binary tumor segmentation
- Model: 2D U-Net
- Loss: BCE + Dice
- Optimizer: AdamW
- Device: Apple MPS when available
- Train: 16,000 slices
- Validation: 4,000 slices

The processed shards are generated by `03_v2_preprocessing.ipynb` and reused here.


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dataset import BraTSSliceDataset
from unet import UNet

BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 10
LEARNING_RATE = 1e-4
THRESHOLD = 0.5

TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "v2" / "train"
VAL_DIR = PROJECT_ROOT / "data" / "processed" / "v2" / "val"
MODEL_DIR = PROJECT_ROOT / "models" / "v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("Train cache exists:", TRAIN_DIR.exists())
print("Validation cache exists:", VAL_DIR.exists())


## 2. Load datasets


In [ ]:
train_dataset = BraTSSliceDataset(TRAIN_DIR)
val_dataset = BraTSSliceDataset(VAL_DIR)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


## 3. Create DataLoaders


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


## 4. Inspect a training batch


In [ ]:
images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print("Mask values:", masks.unique())


## 5. Visualize a multimodal sample


In [ ]:
sample_idx = 0

fig, axes = plt.subplots(1, 5, figsize=(16, 4))

modality_names = ["T1n", "T1c", "T2w", "T2f"]

for i, name in enumerate(modality_names):
    axes[i].imshow(images[sample_idx, i].numpy(), cmap="gray")
    axes[i].set_title(name)
    axes[i].axis("off")

axes[4].imshow(masks[sample_idx].numpy(), cmap="gray")
axes[4].set_title("Tumor Mask")
axes[4].axis("off")

plt.tight_layout()
plt.show()


## 6. Define the U-Net


In [ ]:
model = UNet(
    in_channels=4,
    out_channels=1,
).to(device)

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model:", model.__class__.__name__)
print("Parameters:", f"{num_parameters:,}")
print("Device:", device)


## 7. Loss functions


In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, smooth=1e-6):
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.flatten(1)
    targets = targets.flatten(1)

    intersection = (probabilities * targets).sum(dim=1)

    dice = (
        2 * intersection + smooth
    ) / (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return 1 - dice.mean()


def combined_loss(logits, targets):
    return (
        bce_loss(logits, targets)
        + dice_loss(logits, targets)
    )


def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities > threshold).float()

    predictions = predictions.flatten(1)
    targets = targets.flatten(1)

    intersection = (predictions * targets).sum(dim=1)

    dice = (
        2 * intersection + smooth
    ) / (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return dice.mean().item()


def iou_score(logits, targets, threshold=0.5, smooth=1e-6):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities > threshold).float()

    predictions = predictions.flatten(1)
    targets = targets.flatten(1)

    intersection = (predictions * targets).sum(dim=1)

    union = (
        predictions
        + targets
        - predictions * targets
    ).sum(dim=1)

    iou = (
        intersection + smooth
    ) / (union + smooth)

    return iou.mean().item()


## 8. Optimizer and checkpoint configuration


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

best_val_dice = 0.0

print("Learning rate:", LEARNING_RATE)
print("Checkpoint directory:", MODEL_DIR)


## 9. Single-batch forward/backward test


In [ ]:
test_images, test_masks = next(iter(train_loader))

test_images = test_images.to(device)
test_masks = test_masks.to(device).unsqueeze(1)

optimizer.zero_grad()

test_logits = model(test_images)
test_loss = combined_loss(test_logits, test_masks)

test_loss.backward()
optimizer.zero_grad()

print("Device:", device)
print("Input:", test_images.shape)
print("Target:", test_masks.shape)
print("Logits:", test_logits.shape)
print("Test loss:", round(test_loss.item(), 4))


## 10. Training loop

The best checkpoint is selected using validation Dice.

Two checkpoints are maintained:
- `models/v2/latest.pt`
- `models/v2/best.pt`


In [ ]:
history = {
    "train_loss": [],
    "train_dice": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": [],
    "epoch_time": [],
}

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # -----------------------------
    # Training
    # -----------------------------
    model.train()

    running_loss = 0.0
    running_dice = 0.0

    for batch_images, batch_masks in train_loader:
        batch_images = batch_images.to(device)
        batch_masks = batch_masks.to(device).unsqueeze(1)

        optimizer.zero_grad()

        logits = model(batch_images)
        loss = combined_loss(logits, batch_masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_dice += dice_score(logits, batch_masks, THRESHOLD)

    train_loss = running_loss / len(train_loader)
    train_dice = running_dice / len(train_loader)

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()

    val_running_loss = 0.0
    val_running_dice = 0.0
    val_running_iou = 0.0

    with torch.no_grad():
        for batch_images, batch_masks in val_loader:
            batch_images = batch_images.to(device)
            batch_masks = batch_masks.to(device).unsqueeze(1)

            logits = model(batch_images)
            loss = combined_loss(logits, batch_masks)

            val_running_loss += loss.item()
            val_running_dice += dice_score(
                logits, batch_masks, THRESHOLD
            )
            val_running_iou += iou_score(
                logits, batch_masks, THRESHOLD
            )

    val_loss = val_running_loss / len(val_loader)
    val_dice = val_running_dice / len(val_loader)
    val_iou = val_running_iou / len(val_loader)

    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)
    history["epoch_time"].append(epoch_time)

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_dice": val_dice,
        "val_iou": val_iou,
        "history": history,
    }

    torch.save(
        checkpoint,
        MODEL_DIR / "latest.pt"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice

        torch.save(
            checkpoint,
            MODEL_DIR / "best.pt"
        )

        best_marker = " ← best"
    else:
        best_marker = ""

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"Time: {epoch_time / 60:.2f} min"
        f"{best_marker}"
    )


## 11. Plot training curves


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(7, 5))
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("V2 Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(epochs, history["train_dice"], label="Train Dice")
plt.plot(epochs, history["val_dice"], label="Validation Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("V2 Dice Score")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(epochs, history["val_iou"], label="Validation IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("V2 Validation IoU")
plt.legend()
plt.grid(True)
plt.show()


## 12. Final training summary


In [ ]:
best_epoch = int(np.argmax(history["val_dice"])) + 1
best_dice = max(history["val_dice"])
best_iou = history["val_iou"][best_epoch - 1]

print("Best epoch:", best_epoch)
print("Best validation Dice:", f"{best_dice:.4f}")
print("Validation IoU at best Dice:", f"{best_iou:.4f}")
print("Best checkpoint:", MODEL_DIR / "best.pt")
print("Latest checkpoint:", MODEL_DIR / "latest.pt")


## V2 training result

* **Best validation Dice:** 0.8240
* **Validation IoU:** 0.7919
* **Best epoch:** 10
* **Epoch time:** ~7.04 minutes
* **Device:** Apple MPS
* **Input:** 4-channel 128×128 MRI slice
* **Model:** 2D U-Net
* **Loss:** BCE + Dice

The next notebook will evaluate the best checkpoint qualitatively and quantitatively.

